# KB Card 파일 불러오기

## 0. 초기 세팅
각종 라이브러리를 불러오고 초기값 등을 세팅합니다.

In [1]:
#!uv add xlrd

In [2]:
import pandas as pd
from ledgerly.expenditure import (
    preprocess_kbcard_data,
    map_kb_card_df_to_expenditure,
    map_category,
    insert_expenditure_data,
)
from ledgerly.expenditure import kbcard_file_config

## 1. 데이터 파일 조회 및 전처리

In [3]:
# 데이터 파일 경로 설정 (가변적)
target_yymm = "2603" # 예: "2603", 없을 경우 빈 문자열 ""
target_filename = "kbcard_2602.xls" # 예: "kbcard_2602.xls" 또는 "kbcard.xls"

if target_yymm:
    target_file = kbcard_file_config["data_dir"] / target_yymm / target_filename
else:
    target_file = kbcard_file_config["data_dir"] / target_filename

# 데이터 불러오기
raw_df = pd.read_excel(target_file, header=6)

# 데이터 전처리
df = preprocess_kbcard_data(raw_df)


## 2. DB 데이터로 매핑

In [4]:
expenditure_df = map_kb_card_df_to_expenditure(df)
expenditure_df[expenditure_df["amount"] < 0]

,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid


## 3. 지출 분류

In [5]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")


## 4. DB 저장

In [6]:
insert_expenditure_data(expenditure_df)
print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")

87개의 데이터가 성공적으로 저장되었습니다.
